In [95]:
import pandas as pd

# Charger le CSV brut
df = pd.read_csv(r"C:\Users\km_sa\DEV\m2_enedis_dpe_app\app\data\df_adem_enedis_iris_69_prepared.csv.gz", sep=",")

In [96]:
df["code_postal_ban"].nunique()

52

In [97]:
df.loc[df["code_postal_ban"] == 69250, "nom_commune_ban"].unique()

array(['ALBIGNY-SUR-SAÔNE', 'NEUVILLE-SUR-SAÔNE',
       "POLEYMIEUX-AU-MONT-D'OR", 'MONTANAY', 'FLEURIEU-SUR-SAÔNE',
       'FLEURIEU-SUR-SA*U00F4NE', "CURIS-AU-MONT-D'OR",
       'NEUVILLE-SUR-SAÃ NE', 'NEUVILLE-SUR-SAONE',
       'ALBIGNY-SUR-SA*U00F4NE', 'ALBIGNY-SUR-SAONE', 'ALBIGNY SUR SAONE',
       'NEUVILLE-SUR-SAONNE'], dtype=object)

In [98]:
import unicodedata
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    text = text.upper().strip()
    text = re.sub(r"[^A-Z '-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

In [99]:
import re
from rapidfuzz import fuzz

def normalize_lyon_name(cp, name):
    """
    Corrige automatiquement les noms des arrondissements de Lyon.
    Exemples : 
        'LYON', 'LYON ARRONDISSEMENT', 'LYON ER ARRONDISSEMENT' 
        → 'LYON 1ER ARRONDISSEMENT' (selon le code postal)
    """
    try:
        cp = int(cp)
    except:
        return name

    if 69001 <= cp <= 69009:
        arrondissement = cp - 69000  # Ex: 69003 -> 3
        suffix = "ER" if arrondissement == 1 else "E"
        return f"LYON {arrondissement}{suffix} ARRONDISSEMENT"
    return name


def harmonize_communes(df, threshold=75):
    df["nom_commune_ban_clean"] = df["nom_commune_ban"].apply(clean_text)
    harmonized = []

    for cp, subdf in df.groupby("code_postal_ban"):
        names = subdf["nom_commune_ban_clean"].dropna().unique().tolist()

        # 🧠 Cas spécial : Lyon (69001–69009)
        if 69001 <= int(cp) <= 69009:
            subdf["nom_commune_fuzzy"] = subdf["nom_commune_ban_clean"].apply(lambda x: normalize_lyon_name(cp, x))
            harmonized.append(subdf)
            continue

        # Sinon fuzzy matching classique
        clusters = {}
        for name in names:
            if any(name in g for g in clusters.values()):
                continue
            matches = [n for n in names if fuzz.ratio(name, n) >= threshold]
            clusters[name] = matches

        local_map = {g: min(v, key=len) for v in clusters.values() for g in v}
        subdf["nom_commune_fuzzy"] = subdf["nom_commune_ban_clean"].map(local_map)
        harmonized.append(subdf)

    return pd.concat(harmonized)

In [100]:
df = harmonize_communes(df, threshold=75)

In [101]:
df.loc[df["code_postal_ban"] == 69250, "nom_commune_fuzzy"].unique()

array(['ALBIGNY-SUR-SAONE', 'NEUVILLE-SUR-SAONE',
       "POLEYMIEUX-AU-MONT-D'OR", 'MONTANAY', 'FLEURIEU-SUR-SAONE',
       "CURIS-AU-MONT-D'OR"], dtype=object)

In [102]:
df.loc[df["code_postal_ban"].between(69001, 69009), "nom_commune_fuzzy"].unique()

array(['LYON 1ER ARRONDISSEMENT', 'LYON 2E ARRONDISSEMENT',
       'LYON 3E ARRONDISSEMENT', 'LYON 4E ARRONDISSEMENT',
       'LYON 5E ARRONDISSEMENT', 'LYON 6E ARRONDISSEMENT',
       'LYON 7E ARRONDISSEMENT', 'LYON 8E ARRONDISSEMENT',
       'LYON 9E ARRONDISSEMENT'], dtype=object)

In [103]:
communes_par_cp = (
    df.groupby("code_postal_ban")["nom_commune_fuzzy"]
    .unique()            # communes uniques par code postal
    .apply(list)         # transforme en vraie liste
    .to_dict()           # dictionnaire final
)

In [104]:
for cp, communes in list(communes_par_cp.items()):
    print(cp, "→", communes)

69001 → ['LYON 1ER ARRONDISSEMENT']
69002 → ['LYON 2E ARRONDISSEMENT']
69003 → ['LYON 3E ARRONDISSEMENT']
69004 → ['LYON 4E ARRONDISSEMENT']
69005 → ['LYON 5E ARRONDISSEMENT']
69006 → ['LYON 6E ARRONDISSEMENT']
69007 → ['LYON 7E ARRONDISSEMENT']
69008 → ['LYON 8E ARRONDISSEMENT']
69009 → ['LYON 9E ARRONDISSEMENT']
69100 → ['VILLEURBANNE']
69110 → ['SAINTE-FOY-LES-LYON']
69120 → ['VAULX-EN-VELIN']
69130 → ['ECULLY']
69140 → ['RILLIEUX-LA-PAPE']
69150 → ['DECINES-CHARPIEU']
69160 → ['TASSIN-LA-DEMI-LUNE']
69190 → ['SAINT-FONS']
69200 → ['VENISSIEUX']
69230 → ['SAINT-GENIS-LAVAL']
69250 → ['ALBIGNY-SUR-SAONE', 'NEUVILLE-SUR-SAONE', "POLEYMIEUX-AU-MONT-D'OR", 'MONTANAY', 'FLEURIEU-SUR-SAONE', "CURIS-AU-MONT-D'OR"]
69260 → ['CHARBONNIERES-LES-BAINS']
69270 → ['ROCHETAILLEE-SUR-SAONE', 'FONTAINES-SUR-SAONE', 'FONTAINES-SAINT-MARTIN', "SAINT-ROMAIN-AU-MONT-D'OR", 'CAILLOUX-SUR-FONTAINES', "COUZON-AU-MONT-D'OR"]
69280 → ["MARCY-L'ETOILE"]
69290 → ['CRAPONNE', 'SAINT-GENIS-LES-OLLIERES', 'GREZI

In [106]:
df = df.drop(columns=["nom_commune_ban"], errors="ignore")

df = df.rename(columns={"nom_commune_fuzzy": "nom_commune_ban"})

In [109]:
output_path = r"C:\Users\km_sa\DEV\m2_enedis_dpe_app\app\data\df_adem_enedis_iris_69_prepared.csv.gz"
df.to_csv(output_path, index=False, compression="gzip", encoding="utf-8")
print("Fichier enregistré avec succès :", output_path)

Fichier enregistré avec succès : C:\Users\km_sa\DEV\m2_enedis_dpe_app\app\data\df_adem_enedis_iris_69_prepared.csv.gz
